In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from pymoo.indicators.hv import Hypervolume

from util import natural_key, load_yaml

In [3]:
def load_pareto_history(filepath="pareto_history.pkl"):
    """
    Carga el archivo pickle que contiene fronts_history.
    Devuelve un dict: {generacion: {nivel_frente: [registros...]}}
    """
    with open(filepath, "rb") as f:
        history = pickle.load(f)
    return history

In [4]:
def pareto_history_to_df(filepath="pareto_history.pkl", generation=None):
    """
    Convierte la historia de Pareto guardada en filepath en un DataFrame
    para la generación 'generation', con columnas:
        generation, front_level, accuracy, params, inference_time
    """
    history = load_pareto_history(filepath)
        
    if generation is None:
        generation = max(history.keys())
        
    gen_dict = history[generation]
    records = []
    for level, recs in gen_dict.items():
        # saltamos la métrica hypervolume
        if level == "hypervolume":
            continue
        for rec in recs:
            records.append({
                "id":             rec["id"],
                "generation":      generation,
                "front_level":     level,
                "accuracy":        rec["accuracy"],
                "params":          rec["params"],
                "inference_time":  rec["inference_time"]
            })
    return pd.DataFrame(records)

In [5]:
def archive_to_df(experiment_path: str, archive_subdir: str = "archive") -> pd.DataFrame:
    """Load training metrics from the archive directory into a DataFrame.

    Each folder inside ``archive_subdir`` should contain a ``training_params.txt``
    file with keys such as ``fitness_metric``, ``cuda_inference_time`` and
    ``total_params``. The value for ``fitness_metric`` is taken from the metric
    name specified in that file.

    Parameters
    ----------
    experiment_path : str
        Path to the root experiment directory.
    archive_subdir : str, optional
        Name of the subdirectory that stores archived results. Defaults to
        ``"archive"``.

    Returns
    -------
    pandas.DataFrame
        DataFrame with columns ``id``, ``fitness_metric``, ``cuda_inference_time``
        and ``total_params`` for all archived runs.
    """

    archive_dir = os.path.join(experiment_path, archive_subdir)
    if not os.path.isdir(archive_dir):
        raise FileNotFoundError(f"Archive folder not found at {archive_dir}")

    records = []
    for folder in sorted(os.listdir(archive_dir), key=natural_key):
        params_file = os.path.join(archive_dir, folder, "training_params.txt")
        if not os.path.isfile(params_file):
            continue

        params = load_yaml(params_file)
        metric_key = params.get("fitness_metric")
        if metric_key is None:
            continue

        fitness_value = params.get(metric_key)
        cuda_time = params.get("cuda_inference_time")
        tot_params = params.get("total_params")

        records.append(
            {
                "id": folder,
                "fitness_metric": fitness_value,
                "cuda_inference_time": cuda_time,
                "total_params": tot_params,
            }
        )

    return pd.DataFrame(records)

In [6]:
path_archive = "experiment_cifar10_nsgaX/exp2_repeat_1"
df_archive = archive_to_df(path_archive)
df_archive.shape

(52, 4)

In [7]:
df_archive

,id,fitness_metric,cuda_inference_time,total_params
0,0_18,52.6,339.961052,24554
1,1_16,54.4,432.848930,368554
2,1_18,51.7,344.276428,22570
3,2_1,48.1,2942.276001,7210
4,2_2,56.9,6416.845322,139530
5,2_8,56.6,2736.616135,73706
6,2_12,57.2,1982.021332,380906
7,2_16,47.3,4275.512695,6442
8,2_19,54.6,1820.325851,85994
9,3_16,50.4,525.331497,21098


In [8]:
pkl_path = os.path.join(path_archive, "pareto_history.pkl")
df_history = pareto_history_to_df(pkl_path)
df_history = df_history[df_history["front_level"] == 1]

In [9]:
# check if there are some ids that has the same values in accuracy, params and inference_time, in df_history, if so, remove the ones that are not in df_archive
duplicated_rows = df_history[df_history.duplicated(subset=['accuracy', 'params', 'inference_time'], keep=False)]
if not duplicated_rows.empty:
    print("Duplicated rows found")
    archive_ids = set(df_archive['id'])
    rows_to_remove = duplicated_rows[~duplicated_rows['id'].isin(archive_ids)]
    if not rows_to_remove.empty:
        print(f"Removing {len(rows_to_remove)} rows")
        df_history = df_history.drop(rows_to_remove.index)


In [10]:
df_history

,id,generation,front_level,accuracy,params,inference_time
0,0_18,19,1,52.6,24554.0,339.961052
1,10_19,19,1,27.8,1482.0,7172.250748
2,11_17,19,1,35.7,3786.0,7386.827469
3,11_18,19,1,42.1,6090.0,7814.741135
4,12_1,19,1,45.4,6090.0,14686.012268
5,12_16,19,1,51.8,349258.0,328.612328
6,12_19,19,1,36.1,4490.0,211.620331
7,12_2,19,1,59.6,456394.0,4492.354393
8,14_17,19,1,64.4,692586.0,394.272804
9,14_2,19,1,66.5,488138.0,8716.058731


In [11]:
# check if there are any different ids in the archive and history
archive_ids = set(df_archive["id"])
history_ids = set(df_history["id"])
print(f"Archive IDs: {len(archive_ids)}")
print(f"History IDs: {len(history_ids)}")
print(f"Different IDs: {len(archive_ids - history_ids)}")
# check the values of the ids that are in the archive but not in the history
diff_ids = archive_ids - history_ids
print("Different IDs:", diff_ids)

Archive IDs: 52
History IDs: 52
Different IDs: 0
Different IDs: set()
